In [ ]:
import numpy as np
import os
import pandas as pd
from datetime import date, datetime
import matplotlib.pyplot as plt

### Data treatment
The time series data is divided in intervals of 6 hours to count a relativily ok number of failures
The first code is generating datasets without the failures, for analysis

In [ ]:
original_data_path = 'dataset 14-7/dataset 14-7/datasets vazao/'
final_path = 'datasets/treated throughput datasets 14-07-2024'
protocols = ['bbr', 'cubic']

In [ ]:
# def data_treatment_intervals(destination_data_path, original_data_path, protocols):
#     for protocol in protocols:
#         data_path = original_data_path + "/" + protocol

#         destination_path = destination_data_path + "/" + protocol

#         files = os.listdir(data_path)
#         sorted_files = sorted(files)

#         if not os.path.exists(destination_path):
#             os.makedirs(destination_path)

#         for file_name in sorted_files:
#             dataset = pd.read_csv(os.path.join(data_path, file_name))

#             dataset = dataset.rename(columns={'Vazao': 'Throughput'})

#             dataset['Timestamp'] = pd.to_datetime(dataset['Timestamp'], unit='s')

#             # Define function to get interval start time
#             def get_interval_start(dt):
#                 hour = dt.hour
#                 if hour < 6:
#                     return dt.replace(hour=0, minute=0, second=0, microsecond=0)
#                 elif hour < 12:
#                     return dt.replace(hour=6, minute=0, second=0, microsecond=0)
#                 elif hour < 18:
#                     return dt.replace(hour=12, minute=0, second=0, microsecond=0)
#                 else:
#                     return dt.replace(hour=18, minute=0, second=0, microsecond=0)

#             # Apply the function to create 'Interval' column
#             dataset['Interval'] = dataset['Timestamp'].apply(get_interval_start)

#             def calculate_mean_without_outliers(data):
#                 Q1 = data.quantile(0.25)
#                 Q3 = data.quantile(0.75)
#                 IQR = Q3 - Q1
#                 lower_outlier_cut = Q1 - 1.5 * IQR
#                 upper_outlier_cut = Q3 + 1.5 * IQR
#                 filtered_data = data[(data >= lower_outlier_cut) & (data <= upper_outlier_cut)]
#                 return filtered_data.mean()

#             # Group by 'Interval' and calculate mean of 'Vazao'
#             grouped = dataset.groupby('Interval')['Throughput'].apply(calculate_mean_without_outliers).reset_index()

#             # Format 'Interval' to 'dd-mm-aa hh:mm:ss'
#             grouped['Interval'] = grouped['Interval'].dt.strftime('%d-%m-%y %H:%M:%S')

#             output_file = os.path.join(destination_path, "treated " + file_name)
#             grouped.to_csv(output_file, index=False)

# saving_dir = 'datasets/throughput datasets without failures 14-07-2024'
# original_dir = 'dataset 14-7/dataset 14-7/datasets vazao/'
# protocols = ['bbr', 'cubic']
# data_treatment_intervals(saving_dir, original_dir, protocols)

In [ ]:
# def count_lines_in_files(file_path):
#     with open(file_path, 'r') as f:
#         line_count = sum(1 for _ in f)  # Count lines
#     return line_count

# def compare_directories_by_lines(original_path, final_path):
#     report_data = []

#     for file in os.listdir(final_path):
#         file_path_final = final_path + "/" + file 
#         lines_dir1 = count_lines_in_files(file_path_final)

#         file_name = file.split("treated ")[1]  # Remove the "treated"
#         file_path_original = original_path + "/" + file_name
#         lines_dir2 = count_lines_in_files(file_path_original)

#         diff_percentage = (lines_dir2 - lines_dir1) / lines_dir2 * 100 if lines_dir2 > 0 else 0

#         report_data.append({
#             'File Name': file_name,
#             'Initial Line Count': lines_dir2,
#             'Final Line Count': lines_dir1,
#             'Line Count Difference (%)': diff_percentage
#         })
    
#     report_df = pd.DataFrame(report_data)
#     return report_df

# protocols = ['bbr', 'cubic']
# treated_path = 'datasets/throughput datasets without failures 14-07-2024'
# original_data_path = 'dataset 14-7/dataset 14-7/datasets vazao'

# for protocol in protocols:
#     original_path = original_data_path + "/" + protocol
#     final_path = treated_path + "/" + protocol
#     report_df = compare_directories_by_lines(original_path, final_path)
#     treatment_loss_mean = report_df[report_df['Line Count Difference (%)'] != 0]['Line Count Difference (%)'].mean()
#     print (f'In the {protocol} throughput data processing, close to {treatment_loss_mean:.2f}% of the data was lost.')
#     report_df.to_csv(f'treatment_loss_report {protocol}.csv')


### Adding the failures...
In his approach we decided to use np.nan, not -1

In [ ]:
def data_treatment_intervals(destination_data_path, original_data_path, protocols):
    for protocol in protocols:
        data_path = original_data_path + "/" + protocol

        destination_path = destination_data_path + "/" + protocol

        files = os.listdir(data_path)
        sorted_files = sorted(files)

        if not os.path.exists(destination_path):
            os.makedirs(destination_path)

        

        for file_name in sorted_files:
            dataset = pd.read_csv(os.path.join(data_path, file_name))

            if dataset.empty:
                print(f"No data in {file_name}. Skipping...")
                continue

            dataset = dataset.rename(columns={'Vazao': 'Throughput'})

            dataset['Timestamp'] = pd.to_datetime(dataset['Timestamp'], unit='s')

            # Define function to get interval start time
            def get_interval_start(dt):
                hour = dt.hour
                if hour < 6:
                    return dt.replace(hour=0, minute=0, second=0, microsecond=0)
                elif hour < 12:
                    return dt.replace(hour=6, minute=0, second=0, microsecond=0)
                elif hour < 18:
                    return dt.replace(hour=12, minute=0, second=0, microsecond=0)
                else:
                    return dt.replace(hour=18, minute=0, second=0, microsecond=0)

            # Apply the function to create 'Interval' column
            dataset['Timestamp'] = dataset['Timestamp'].apply(get_interval_start)

            def calculate_mean_without_outliers(data):
                Q1 = data.quantile(0.25)
                Q3 = data.quantile(0.75)
                IQR = Q3 - Q1
                lower_outlier_cut = Q1 - 1.5 * IQR
                upper_outlier_cut = Q3 + 1.5 * IQR
                filtered_data = data[(data >= lower_outlier_cut) & (data <= upper_outlier_cut)]
                return filtered_data.mean()

            # Group by 'Interval' and calculate mean of 'Vazao'
            grouped = dataset.groupby('Timestamp')['Throughput'].apply(calculate_mean_without_outliers).reset_index()

            start_time = dataset['Timestamp'].min().replace(hour=0, minute=0, second=0, microsecond=0)
            end_time = dataset['Timestamp'].max().replace(hour=18, minute=0, second=0, microsecond=0)
            all_intervals = pd.date_range(start=start_time, end=end_time, freq='6h')

            grouped = grouped.set_index('Timestamp').reindex(all_intervals).reset_index()
            grouped.columns = ['Timestamp', 'Throughput']  # Rename columns after reindex
            grouped['Throughput'] = grouped['Throughput'].fillna(np.nan)

            # Format 'Interval' to 'dd-mm-aa hh:mm:ss'
            grouped['Timestamp'] = grouped['Timestamp'].dt.strftime('%d-%m-%y %H:%M:%S')

            output_file = os.path.join(destination_path, "treated " + file_name)
            grouped.to_csv(output_file, index=False)

saving_dir = 'datasets/treated throughput datasets 14-07-2024'
original_dir = 'dataset 14-7/dataset 14-7/datasets vazao/'
protocols = ['bbr', 'cubic']
data_treatment_intervals(saving_dir, original_dir, protocols)

Checking how many data we lost by cleaning outliers and applying interval 

In [ ]:
def plot_original(df, ax, name, column_name='Vazao', time_column_name='Timestamp', point_size = 2, point_color = 'red'):
    df[time_column_name] = pd.to_datetime(df[time_column_name])
    df = df.sort_values(by=time_column_name)

    ax.scatter(df[time_column_name], df[column_name], s=point_size, c=point_color)  
    ax.set_xlabel('Timestamp')
    ax.set_ylabel('Throughput')
    ax.set_title(name)
    ax.tick_params(axis='x', rotation=45)

def generate_plots(link, original_data_path, treated_path):
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'Throughput by Time for {link}', fontsize=16)

    # Original BBR
    df = pd.read_csv(f'{original_data_path}/bbr/bbr esmond data {link} 07-13-2024.csv')
    plot_original(df, axs[0, 0], f'Original BBR {link}')

    # Original Cubic
    df2 = pd.read_csv(f'{original_data_path}/cubic/cubic esmond data {link} 07-13-2024.csv')
    plot_original(df2, axs[0, 1], f'Original Cubic {link}')

    # Treated BBR
    df = pd.read_csv(f'{treated_path}/bbr/treated bbr esmond data {link} 07-13-2024.csv')
    df = df.dropna(subset=['Throughput'])
    plot_original(df, axs[1, 0], f'Treated BBR {link}', 'Throughput')

    # Treated Cubic
    df2 = pd.read_csv(f'{treated_path}/cubic/treated cubic esmond data {link} 07-13-2024.csv')
    df2 = df2.dropna(subset=['Throughput']) 
    plot_original(df2, axs[1, 1], f'Treated Cubic {link}', 'Throughput')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  
    plt.show()

links = ['es-sp', 'ms-to', 'pb-ba']
original_data_path = 'dataset 14-7/dataset 14-7/datasets vazao/'
final_path = 'datasets/treated throughput datasets 14-07-2024'

for link in links:
    generate_plots(link, original_data_path, final_path)


### Dataset analysis
Analysing failure rate and size of datasets of 14-7-2024 collection with MonIpê Network Monitoring Tool

In [ ]:
import os
import pandas as pd

def get_longest_interval(treated_throughput_path, saving_path = 'longest interval', quantity=10):
    all_files = []

    for protocol in os.listdir(treated_throughput_path):
        folder_path = os.path.join(treated_throughput_path, protocol)
        if os.path.isdir(folder_path):  # Ensure it's a directory
            for file in os.listdir(folder_path):
                file_path = os.path.join(folder_path, file)
                if file.endswith('.csv'):
                    df = pd.read_csv(file_path)

                    longest_interval = []
                    current_interval = []

                    for _, row in df.iterrows():
                        if not pd.isna(row['Throughput']):  # Check for non-NaN values
                            current_interval.append(row)
                        else:
                            if len(current_interval) > len(longest_interval):
                                longest_interval = current_interval
                            current_interval = []

                    # Final check after loop to capture the last interval
                    if len(current_interval) > len(longest_interval):
                        longest_interval = current_interval

                    # Uncomment for saving the longest interval if it meets criteria (optional)
                    # if len(longest_interval) >= 10:
                    #     longest_df = pd.DataFrame(longest_interval)
                    #     output_file = os.path.splitext(file)[0] + '_longest_interval.csv'
                    #     longest_df.to_csv(os.path.join(saving_path, output_file), index=False)

                    # Store file and interval information
                    all_files.append({'file': file, 'interval_length': len(longest_interval)})

    # Sort by interval length and return the top files based on quantity
    sorted_files = sorted(all_files, key=lambda x: x['interval_length'], reverse=True)
    return sorted_files[:quantity]


In [ ]:
get_longest_interval(final_path)

In [ ]:
def get_files_size_failure_rate(percentage, path, protocols, quantity = 10):
    archive_data = {}

    # Loop through all relevant CSV files in the directory
    for protocol in protocols:
        full_path = os.path.join(path, protocol)
        for arquivo in os.listdir(full_path):
            if arquivo.endswith('.csv'):
                caminho_arquivo = os.path.join(full_path, arquivo)
                df = pd.read_csv(caminho_arquivo)

                total_rows = len(df)
                num_failures = df['Throughput'].isna().sum()
                failure_percentage = (num_failures / total_rows) * 100 if total_rows > 0 else 0
                
                # Include only archives with failure percentage below specified percentage
                if failure_percentage < percentage:
                    # Store the DataFrame, total rows, and failure percentage in the dictionary
                    archive_data[f'{protocol} {arquivo}'] = {'df': df, 'total_rows': total_rows, 'failure_percentage': failure_percentage}

        # Check if any archives meet the criteria
        if not archive_data:
            print(f"No archives with failure percentage below {percentage}% were found.")
            return []

        # Sort the archives by line count in descending order
        sorted_archives = sorted(
            archive_data.items(),
            key=lambda x: x[1]['total_rows'],
            reverse=True
        )


    top_archives = sorted_archives[:quantity]

    print(f"Top {quantity} archives with failure percentage below {percentage}%:")
    for archive, data in top_archives:
        failure_percentage = data['failure_percentage']
        total_rows = data['total_rows']
        print(f"{archive}: {total_rows} lines, Failure Percentage: {failure_percentage:.2f}%")

    return top_archives


In [ ]:
for protocol in protocols:
    get_files_size_failure_rate(30, final_path, protocols)